In [114]:
from pathlib import Path
import numpy as np
import pandas as pd
import random

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

import asyncio
import json
from ollama import AsyncClient
from tqdm.asyncio import tqdm_asyncio
import pyarrow

In [10]:
file_path = Path.cwd().parent / "dataset" / "processed" / "amazon_tweets_english_v1.parquet"
final_amazon_df = pd.read_parquet(file_path)

In [11]:
final_amazon_df.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,conversation_id
0,617,115820,True,Tue Oct 31 22:16:32 +0000 2017,Way to drop the ball on customer service @1158...,615,NaN,2
1,615,AmazonHelp,False,Tue Oct 31 22:29:00 +0000 2017,@115820 I'm sorry we've let you down! Without ...,616,617.0,2
2,616,115820,True,Tue Oct 31 23:22:08 +0000 2017,@AmazonHelp 3 different people have given 3 di...,618,615.0,2
3,618,AmazonHelp,False,Tue Oct 31 23:28:00 +0000 2017,@115820 We'd like to take a further look into ...,619,616.0,2
4,619,115820,True,Tue Oct 31 23:32:26 +0000 2017,@AmazonHelp I frankly don't have the patience ...,NaN,618.0,2


In [12]:
final_amazon_df.shape

(280081, 8)

In [13]:
final_amazon_df["conversation_id"].nunique()

61850

In [14]:
# 1. Calculate how many messages are in each conversation thread
convo_lengths = final_amazon_df['conversation_id'].value_counts()

# 2. Keep only conversations that have at least 2 messages (Customer starter + Amazon reply)
# This is crucial for RAG, as a 1-message thread gives no example of how the brand replies.
valid_convos = convo_lengths[convo_lengths >= 2].index.tolist()

# 3. Randomly sample exactly 15,000 unique conversation IDs
target_size = 15000
random.seed(42)  # For reproducible sampling across your notebooks
sampled_convo_ids = random.sample(valid_convos, min(target_size, len(valid_convos)))

# 4. Filter the main dataframe to lock in your final modeling subset
# This dataframe contains the full, unbroken chats for your RAG system
rag_master_df = final_amazon_df[final_amazon_df['conversation_id'].isin(sampled_convo_ids)].copy()
rag_master_df = rag_master_df.reset_index(drop=True)

print(f"Total Rows Saved for RAG/Training: {rag_master_df.shape[0]}")
print(f"Total Unique Conversations: {rag_master_df['conversation_id'].nunique()}")

Total Rows Saved for RAG/Training: 66995
Total Unique Conversations: 15000


# Classifier dataset extraction - 

In [15]:
# Extract only the opening message of each of the 15,000 conversations
classifier_df = rag_master_df[
    (rag_master_df['inbound'] == True) & 
    (rag_master_df['in_response_to_tweet_id'].isna())
].copy()

# In case some threads started mid-conversation due to missing upstream raw data,
# we pick the earliest chronological message of that thread as a fallback starter.
missing_starters = set(sampled_convo_ids) - set(classifier_df['conversation_id'])
if missing_starters:
    fallback_df = rag_master_df[rag_master_df['conversation_id'].isin(missing_starters)]
    fallback_starters = fallback_df.sort_values('created_at').groupby('conversation_id').first()
    classifier_df = pd.concat([classifier_df, fallback_starters]).drop_duplicates(subset=['tweet_id'])

classifier_df = classifier_df.reset_index(drop=True)
print(f"Total Clean Customer Starters for Classifier: {len(classifier_df)}")

Total Clean Customer Starters for Classifier: 15000


In [16]:
processed_dir = Path.cwd().parent / "dataset" / "processed"

In [17]:
print(processed_dir)

D:\Projects\Customer-Support-Agent\dataset\processed


In [18]:
rag_output_file = processed_dir / 'amazon_rag_dataset_v1.parquet'
classifier_output_file = processed_dir / 'amazon_classifier_dataset_v1.parquet'

In [19]:
rag_master_df = rag_master_df.reset_index(drop=True)
classifier_df = classifier_df.reset_index(drop=True)

In [20]:
rag_master_df.to_parquet(rag_output_file, compression='snappy', index=False)
classifier_df.to_parquet(classifier_output_file, compression='snappy', index=False)

print("--- Pipeline Assets Saved Successfully ---")
print(f"RAG Vector Base saved to:       {rag_output_file.name} ({rag_master_df.shape})")
print(f"Classifier Input saved to:      {classifier_output_file.name} ({classifier_df.shape})")

--- Pipeline Assets Saved Successfully ---
RAG Vector Base saved to:       amazon_rag_dataset_v1.parquet ((66995, 8))
Classifier Input saved to:      amazon_classifier_dataset_v1.parquet ((15000, 8))


In [30]:
# Extract the top 40 most common 2-to-3 word combinations
cv = CountVectorizer(ngram_range=(2, 3), stop_words='english', max_features=40)
counts = cv.fit_transform(classifier_df['text'].dropna())
phrases = pd.DataFrame(counts.sum(axis=0).T, index=cv.get_feature_names_out(), columns=['frequency'])

print("--- Top Customer Opening Phrases ---")
print(phrases.sort_values(by='frequency', ascending=False))

--- Top Customer Opening Phrases ---
                     frequency
customer service           588
amazon prime               379
day delivery               377
day shipping               301
amazonhelp hi              264
115850 amazonhelp          259
115821 amazonhelp          234
hey 115821                 192
prime membership           183
customer care              174
amazonhelp 115850          157
prime member               152
delivery date              150
115850 order               148
115821 prime               143
amazon pay                 126
amazonhelp order           123
amazonhelp 115821          119
amazonhelp https           116
amazonhelp ordered         114
delivered today            114
115821 https               113
115850 ordered             104
don know                   102
amazon india               101
115850 115821              100
gift card                   99
prime day                   97
package delivered           94
placed order                93
de

In [36]:
unique_convos = list(rag_master_df['conversation_id'].unique())

# 2. Split the conversation IDs: 80% Train, 20% Temporary (Val + Test)
train_ids, temp_ids = train_test_split(unique_convos, test_size=0.20, random_state=42)

# 3. Split the temporary IDs equally: 10% Val, 10% Test
val_ids, test_ids = train_test_split(temp_ids, test_size=0.50, random_state=42)

# Convert to sets for lightning-fast lookup operations
train_set, val_set, test_set = set(train_ids), set(val_ids), set(test_ids)

print(f"Split distribution (Threads): Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")

Split distribution (Threads): Train: 12000 | Val: 1500 | Test: 1500


In [37]:
# Create Aligned RAG Datasets (Full histories)
rag_train = rag_master_df[rag_master_df['conversation_id'].isin(train_set)].reset_index(drop=True)
rag_val   = rag_master_df[rag_master_df['conversation_id'].isin(val_set)].reset_index(drop=True)
rag_test  = rag_master_df[rag_master_df['conversation_id'].isin(test_set)].reset_index(drop=True)

# Create Aligned Intent Classifier Datasets (Customer starters only)
clf_train = classifier_df[classifier_df['conversation_id'].isin(train_set)].reset_index(drop=True)
clf_val   = classifier_df[classifier_df['conversation_id'].isin(val_set)].reset_index(drop=True)
clf_test  = classifier_df[classifier_df['conversation_id'].isin(test_set)].reset_index(drop=True)


In [41]:
processed_dir

WindowsPath('D:/Projects/Customer-Support-Agent/dataset/processed')

In [42]:
# Save RAG splits
rag_train.to_parquet(processed_dir / 'rag_train_v1.parquet', compression='snappy', index=False)
rag_val.to_parquet(processed_dir / 'rag_val_v1.parquet', compression='snappy', index=False)
rag_test.to_parquet(processed_dir / 'rag_test_v1.parquet', compression='snappy', index=False)

# Save Classifier splits
clf_train.to_parquet(processed_dir / 'clf_train_v1.parquet', compression='snappy', index=False)
clf_val.to_parquet(processed_dir / 'clf_val_v1.parquet', compression='snappy', index=False)
clf_test.to_parquet(processed_dir / 'clf_test_v1.parquet', compression='snappy', index=False)

In [44]:
# Extract the top 40 most common 2-to-3 word combinations from the training starters
cv = CountVectorizer(ngram_range=(2, 3), stop_words='english', max_features=40)
counts = cv.fit_transform(clf_train['text'].dropna())
phrases = pd.DataFrame(counts.sum(axis=0).T, index=cv.get_feature_names_out(), columns=['frequency'])

print("--- Top Customer Opening Phrases (Training Split) ---")
print(phrases.sort_values(by='frequency', ascending=False))

--- Top Customer Opening Phrases (Training Split) ---
                     frequency
customer service           460
day delivery               295
amazon prime               291
day shipping               239
amazonhelp hi              206
115850 amazonhelp          205
115821 amazonhelp          179
hey 115821                 159
prime membership           142
prime member               125
amazonhelp 115850          124
customer care              124
115821 prime               117
115850 order               115
delivery date              113
amazon pay                  98
amazonhelp https            98
amazonhelp order            95
delivered today             94
amazonhelp 115821           92
amazonhelp ordered          92
115821 https                86
prime day                   82
don know                    80
package delivered           79
placed order                76
gift card                   75
115850 ordered              73
amazon india                73
cancel order    

In [56]:
import asyncio
import json
from ollama import AsyncClient

# 1. Define the pristine schema of intent classes including your safety guardrails
INTENT_CLASSES = [
    "delivery_late",
    "delivery_missing",
    "delivery_service_complaint",
    "order_cancellation",
    "order_tracking_inquiry",
    "prime_subscription_issue",
    "prime_billing_complaint",
    "amazon_pay_fintech",
    "gift_card_problems",
    "amazon_india_support",
    "general_agent_escalation",  # For angry, highly emotional, or complex human intervention requests
    "other"                     # For random, out-of-scope, or completely irrelevant user messages
]

SYSTEM_PROMPT = f"""
You are an expert customer service routing engine for Amazon. 
Analyze the customer's initial tweet and classify it into EXACTLY ONE of the following intent categories:
{json.dumps(INTENT_CLASSES, indent=2)}

Rules:
- Select 'general_agent_escalation' if the user is extremely angry, threatening legal action, or demanding a human manager immediately.
- Select 'other' if the text is random, completely out-of-scope, gibberish, or unrelated to Amazon business.
- Respond ONLY with a valid JSON object containing a single key "intent". Do not include explanations.

Example output:
{{"intent": "delivery_late"}}
"""

# 2. Asynchronous task worker with a Semaphore lock to protect 6GB VRAM
sem = asyncio.Semaphore(3)  # Maximum 3 concurrent requests to prevent OOM on 6GB VRAM

async def label_single_text(text: str) -> str:
    async with sem:
        try:
            response = await AsyncClient().generate(
                model='qwen3:8b',  
                prompt=f"Classify this text:\n{text}",
                system=SYSTEM_PROMPT,
                options={"temperature": 0.0}, # Deterministic outputs
                format='json'
            )
            # Parse the JSON safe block
            result = json.loads(response['response'])
            predicted_intent = result.get('intent', 'other')
            
            # Guard against the model hallucinating an unlisted class
            if predicted_intent not in INTENT_CLASSES:
                return 'other'
            return predicted_intent
        except Exception as e:
            # Fallback safely to human escalation if the model calls fail
            print(e)
            return 'general_agent_escalation'

async def process_dataset(df: pd.DataFrame):
    # Process text lists concurrently
    tasks = [label_single_text(text) for text in df['text']]
    return await asyncio.gather(*tasks)

# --- Execution Gate ---
# To test this safely without waiting hours, execute it on a small slice first
test_slice = clf_train.head(100).copy()

# Run the async loop inside Jupyter smoothly
labels = await process_dataset(test_slice)
test_slice['predicted_intent'] = labels

print(test_slice[['text', 'predicted_intent']].head(20))

                                                 text  \
0   @AmazonHelp delivery I paid for today,didn’t a...   
1   I'm NEVER using Amazon again! After waiting in...   
2   @AmazonHelp where can I chat with a support me...   
3   This is the Japanese vampire book I blurbed. D...   
4   @AmazonHelp i reset my password 3 times and it...   
5   So sad @AmazonHelp @115830 ruined his FIRST Ha...   
6   @AmazonHelp Im not receiving 2step code by tex...   
7   To the person or persons that stole that big, ...   
8   Hey @115821, maybe don't ask people to pay $8 ...   
9   I know it’s nearly xmas but seriously I ordere...   
10  Worst experience in shopping no product no ref...   
11  @115821 , #fake I phone 7 received from amazon...   
12  Please explain this @AmazonHelp https://t.co/V...   
13  @117093 Hey yall. Why is my checkout not worki...   
14  @AmazonHelp Hi is there a way to confirm that ...   
15  Hey @115821 What’s the point of preordering a ...   
16  @115821 u just lost a prime

In [83]:
fp = Path.cwd().parent / "intent_labelling.txt"

with open(fp, "w", encoding="utf-8") as f:
    f.write(test_slice.to_markdown(index=False))

# Intent Labelling - 

In [89]:
# 1. Finalized, production-grade 13-class taxonomy
INTENT_CLASSES = [
    "delivery_late",
    "delivery_missing",
    "delivery_service_complaint",
    "order_cancellation",
    "order_tracking_inquiry",
    "prime_subscription_issue",
    "prime_billing_complaint",
    "amazon_pay_fintech",
    "gift_card_problems",
    "amazon_india_support",
    "account_and_login_issues",    # NEW: Captures password, login, 2FA bugs
    "device_and_technical_support", # NEW: Captures Echo/FireTV hardware & website errors
    "general_agent_escalation",    # Guardrail: Heavy anger, legal threats, agent requests
    "other"                         # Guardrail: Absolute gibberish, out-of-scope text
]

SYSTEM_PROMPT = f"""
You are an expert customer service routing engine for Amazon.
Analyze the customer's initial tweet and classify it into EXACTLY ONE of the following intent categories:
{json.dumps(INTENT_CLASSES, indent=2)}

Strict Guidelines:
- Select 'account_and_login_issues' for password resets, 2-step codes, and login issues.
- Select 'device_and_technical_support' for Echo/Alexa, Fire TV, Kindle bugs, or website/checkout errors.
- Select 'general_agent_escalation' if the user is extremely angry, cursing, or demanding a human manager immediately.
- Select 'other' ONLY if the text is completely random, an unrelated product recommendation, or total gibberish.
- Respond ONLY with a valid JSON object containing a single key "intent". Do not include explanations.

Example output:
{{"intent": "delivery_late"}}
"""

# 2. Strict traffic cop for 6GB VRAM to protect against OOM crashes
sem = asyncio.Semaphore(3) 

async def label_single_text(client: AsyncClient, text: str) -> str:
    async with sem:
        try:
            response = await client.generate(
                model='qwen3:8b', 
                prompt=f"Classify this text:\n{text}",
                system=SYSTEM_PROMPT,
                options={"temperature": 0.0},
                format='json'
            )
            result = json.loads(response['response'])
            predicted_intent = result.get('intent', 'other')
            return predicted_intent if predicted_intent in INTENT_CLASSES else 'other'
        except Exception:
            # Safe operational fallback
            return 'general_agent_escalation'

async def process_dataset_in_batches(df: pd.DataFrame, save_path: Path):
    client = AsyncClient()
    texts = df['text'].tolist()
    labels = []
    
    # Pack the asynchronous tasks with an elegant progress tracking bar
    tasks = [label_single_text(client, text) for text in texts]
    
    print(f"Starting async classification pipeline for {len(tasks)} queries...")
    labels = await tqdm_asyncio.gather(*tasks)
    
    # Assign back to dataset safely
    df['predicted_intent'] = labels
    
    # Save a complete checkpoint instantly
    df.to_parquet(save_path, compression="snappy", index=False)
    print(f"\n[SUCCESS] All queries processed and saved securely to: {save_path.name}")
    return df

# --- Execution Entry Point ---1
processed_dir = Path.cwd().parent / 'dataset' / 'processed'
output_file_path = processed_dir / 'clf_train_labeled_v1.parquet'

# Rerun the async event engine over your complete dataset
clf_train_labeled = await process_dataset_in_batches(clf_train.copy(), output_file_path)

Starting async classification pipeline for 11349 queries...


100%|██████████| 11349/11349 [2:29:35<00:00,  1.26it/s] 



[SUCCESS] All queries processed and saved securely to: clf_train_labeled_v1.parquet


In [93]:
%debug

> d:\projects\customer-support-agent\venv\lib\site-packages\pandas\compat\_optional.py(161)import_optional_dependency()
    159     except ImportError as err:
    160         if errors == "raise":
--> 161             raise ImportError(msg) from err
    162         return None
    163 



ipdb>  q


In [108]:
clf_train_labeled["predicted_intent"].value_counts()*100 /11349

predicted_intent
delivery_service_complaint      25.623403
other                           20.266103
delivery_late                   14.115781
general_agent_escalation         5.991717
delivery_missing                 5.586395
device_and_technical_support     5.110582
prime_subscription_issue         4.564279
order_tracking_inquiry           4.537845
order_cancellation               3.921050
prime_billing_complaint          2.387876
amazon_india_support             2.343819
amazon_pay_fintech               2.317385
account_and_login_issues         2.229271
gift_card_problems               1.004494
Name: count, dtype: float64

In [113]:
display(clf_train_labeled.head())
clf_train_labeled.shape

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,conversation_id,predicted_intent
0,664,115842,True,Tue Oct 31 21:50:49 +0000 2017,"@AmazonHelp delivery I paid for today,didn’t a...",663,NaN,15.0,delivery_late
1,672,115844,True,Tue Oct 31 20:59:45 +0000 2017,I'm NEVER using Amazon again! After waiting in...,"671,673,674",NaN,18.0,delivery_service_complaint
2,1710,116084,True,Tue Oct 31 22:33:35 +0000 2017,@AmazonHelp where can I chat with a support me...,1709,NaN,28.0,prime_billing_complaint
3,1753,116096,True,Sun Oct 29 18:00:47 +0000 2017,This is the Japanese vampire book I blurbed. D...,1752,NaN,37.0,other
4,3729,116611,True,Tue Oct 31 22:34:33 +0000 2017,@AmazonHelp i reset my password 3 times and it...,3727,NaN,51.0,account_and_login_issues


(11349, 9)

In [118]:
val_output_file_path = processed_dir / 'clf_val_labeled_v1.parquet'
test_output_file_path = processed_dir / 'clf_test_labeled_v1.parquet'

# 2. Run the async loop over the Validation split (~1.5k rows | ~10-12 mins)
print("--- Launching Validation Split Pipeline ---")
clf_val_labeled = await process_dataset_in_batches(clf_val.copy(), val_output_file_path)

# 3. Run the async loop over the Test split (~1.5k rows | ~10-12 mins)
print("\n--- Launching Test Split Pipeline ---")
clf_test_labeled = await process_dataset_in_batches(clf_test.copy(), test_output_file_path)

print("\n--- All Evaluation Splits Frozen to Disk Successfully! ---")

--- Launching Validation Split Pipeline ---
Starting async classification pipeline for 1419 queries...


100%|██████████| 1419/1419 [18:51<00:00,  1.25it/s]



[SUCCESS] All queries processed and saved securely to: clf_val_labeled_v1.parquet

--- Launching Test Split Pipeline ---
Starting async classification pipeline for 1434 queries...


100%|██████████| 1434/1434 [18:58<00:00,  1.26it/s]


[SUCCESS] All queries processed and saved securely to: clf_test_labeled_v1.parquet

--- All Evaluation Splits Frozen to Disk Successfully! ---


In [122]:
clf_val_labeled["predicted_intent"].value_counts()

predicted_intent
delivery_service_complaint      388
other                           268
delivery_late                   203
general_agent_escalation         85
device_and_technical_support     74
delivery_missing                 72
prime_subscription_issue         70
order_tracking_inquiry           68
order_cancellation               41
amazon_india_support             40
account_and_login_issues         38
amazon_pay_fintech               32
prime_billing_complaint          23
gift_card_problems               17
Name: count, dtype: int64

In [123]:
clf_val_labeled.sample(10)

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,conversation_id,predicted_intent
1148,2462175,206941,True,Wed Nov 15 08:07:48 +0000 2017,@AmazonHelp There is supposed to be a deal for...,"2462174,2462176",NaN,66247.0,other
422,707367,289107,True,Tue Oct 10 18:51:21 +0000 2017,@AmazonHelp Plzz help me i buyed my power bank...,707366,NaN,23274.0,other
902,1727747,522476,True,Tue Nov 07 05:46:27 +0000 2017,"So, the nipple clamps I ordered have been “arr...",1727745,NaN,51839.0,delivery_late
1274,2731188,680235,True,Mon Nov 20 15:29:24 +0000 2017,@115830 I'd like to talk about a product that ...,2731187,NaN,73795.0,delivery_missing
696,1278029,153416,True,Thu Oct 26 18:59:11 +0000 2017,Still there is no action taken about service i...,"1278028,1278030,1278031,1278032,1278033,1278034",NaN,39272.0,general_agent_escalation
196,361016,201483,True,Sun Oct 08 17:02:13 +0000 2017,"Hey, @115821 what's the point of us paying for...",361014,NaN,11818.0,delivery_late
870,1638693,501084,True,Tue Oct 17 13:12:05 +0000 2017,What's wrong with @115850? @182822 's India's ...,1638692,NaN,49632.0,other
1290,2772362,159383,True,Mon Nov 27 00:30:46 +0000 2017,@115821 Can you explain how two day shipping i...,2772361,NaN,75005.0,delivery_service_complaint
65,141997,148044,True,Fri Nov 24 11:50:53 +0000 2017,Can someone explain to me why when I'm not log...,141996,NaN,3701.0,order_tracking_inquiry
501,842360,320348,True,Fri Oct 20 13:24:59 +0000 2017,@4822 My account has been suspended. I sent an...,842359,NaN,27330.0,account_and_login_issues
